In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import torch
import evaluate

from datasets import load_dataset
from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)

from sklearn.model_selection import GroupShuffleSplit

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [ ]:
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
dataset = load_dataset("raquiba/Sarcasm_News_Headline")

df = pd.DataFrame(dataset["train"])

df = df[["headline", "is_sarcastic"]]

df.columns = ["text", "label"]

In [ ]:
import re

# TEXT CLEANING FUNCTION


def clean_text(text):

    # lowercase
    text = text.lower()

    # remove urls
    text = re.sub(r"http\S+|www\S+", "", text)

    # remove html tags
    text = re.sub(r"<.*?>", "", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text)

    # remove strange symbols
    text = re.sub(r"[^a-zA-Z0-9!?.,' ]", "", text)

    # remove repeated punctuation
    text = re.sub(r"([!?.,])\1+", r"\1", text)

    # strip spaces
    text = text.strip()

    return text


# Apply cleaning
df["text"] = df["text"].apply(clean_text)

In [ ]:
# REMOVE DUPLICATES


before = len(df)

df = df.drop_duplicates(subset=["text"])

after = len(df)

print("Removed duplicates:", before - after)
print("Remaining samples:", after)

In [ ]:
# REMOVE VERY SHORT TEXT


df = df[df["text"].str.split().str.len() >= 4]

print("Dataset size after short-text removal:", len(df))

In [ ]:
# REMOVE VERY LONG TEXT

df = df[df["text"].str.split().str.len() <= 40]

print("Dataset size after long-text filtering:", len(df))

In [ ]:
print(df["label"].value_counts())

In [ ]:
from sklearn.utils import resample

majority = df[df.label == 0]
minority = df[df.label == 1]

minority_upsampled = resample(
    minority, replace=True, n_samples=len(majority), random_state=42
)

df = pd.concat([majority, minority_upsampled])

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
df["group"] = df["label"].astype(str) + "_" + (df.index // 5).astype(str)

groups = df["group"]

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)

train_idx, temp_idx = next(gss.split(df, groups=groups))

train_df = df.iloc[train_idx]
temp_df = df.iloc[temp_idx]

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED)

val_idx, test_idx = next(gss2.split(temp_df, groups=temp_df["group"]))

val_df = temp_df.iloc[val_idx]
test_df = temp_df.iloc[test_idx]

In [ ]:
dataset = DatasetDict(
    {
        "train": Dataset.from_pandas(train_df[["text", "label"]]),
        "validation": Dataset.from_pandas(val_df[["text", "label"]]),
        "test": Dataset.from_pandas(test_df[["text", "label"]]),
    }
)

In [ ]:
model_name = "roberta-base"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
def tokenize_function(example):

    return tokenizer(example["text"], truncation=True, padding=True, max_length=192)

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="macro"
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    learning_rate=1.5e-5,
    weight_decay=0.01,
    warmup_steps=int(len(dataset["train"]) * 2 / (16 * 2) * 0.15),
    lr_scheduler_type="cosine",
    label_smoothing_factor=0.0,
    max_grad_norm=1.0,
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    save_total_limit=2,
    seed=42,
    report_to="none",
)

In [ ]:
tokenized_dataset = dataset.map(tokenize_function, batched=True)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

In [ ]:
trainer.train()

In [ ]:
SAVE_PATH = "/content/robertabase_sarcasm_model"

trainer.save_model(SAVE_PATH)

tokenizer.save_pretrained(SAVE_PATH)

print("Model Saved!")

In [ ]:
!zip -r robertabase_sarcasm_model.zip /content/robertabase_sarcasm_model

In [ ]:
from google.colab import files

files.download("robertabase_sarcasm_model.zip")